In [0]:
%pip install nltk scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import nltk
from nltk.corpus import reuters
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np
import pandas as pd

# Download Reuters corpus
nltk.download('reuters')

# Task 1: Basic Text Representation
documents = reuters.fileids()
corpus = [reuters.raw(doc_id) for doc_id in documents]

print(f"Number of documents in Reuters corpus: {len(documents)}")
print("Sample document ID:", documents[0])
print("Sample document text:\n", corpus[0][:500])  # Print first 500 chars

[nltk_data] Downloading package reuters to
[nltk_data]     /home/spark-0599cdca-d7d8-440d-9eea-b9/nltk_data...


Number of documents in Reuters corpus: 10788
Sample document ID: test/14826
Sample document text:
 ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT
  Mounting trade friction between the
  U.S. And Japan has raised fears among many of Asia's exporting
  nations that the row could inflict far-reaching economic
  damage, businessmen and officials said.
      They told Reuter correspondents in Asian capitals a U.S.
  Move against Japan might boost protectionist sentiment in the
  U.S. And lead to curbs on American imports of their products.
      But some exporters said that while the conflict wo


In [0]:

# Task 2: Bag-of-Words Representation
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(corpus)

print(f"\nBag-of-Words vocabulary size: {len(count_vectorizer.vocabulary_)}")
print(f"Bag-of-Words matrix shape: {bow_matrix.shape}")

# Show first 10 terms in sorted vocabulary
sorted_vocab = sorted(count_vectorizer.vocabulary_.items(), key=lambda x: x[1])
print("First 10 terms in vocabulary:", [term for term, idx in sorted_vocab[:10]])


Bag-of-Words vocabulary size: 30916
Bag-of-Words matrix shape: (10788, 30916)
First 10 terms in vocabulary: ['00', '000', '0000', '00000', '0009', '001', '002', '003', '0037', '004']


In [0]:
# Task 3: TF-IDF Representation
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print(f"\nTF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# Compare first document in both representations
bow_features = bow_matrix[0].nonzero()[1]
tfidf_features = tfidf_matrix[0].nonzero()[1]
print(f"Number of non-zero features in first doc (BoW): {len(bow_features)}")
print(f"Number of non-zero features in first doc (TF-IDF): {len(tfidf_features)}")

# Task 4: Stop Words Analysis
tfidf_vectorizer_sw = TfidfVectorizer(stop_words='english')
tfidf_matrix_sw = tfidf_vectorizer_sw.fit_transform(corpus)

print(f"\nTF-IDF (stop words removed) vocabulary size: {len(tfidf_vectorizer_sw.vocabulary_)}")
print(f"TF-IDF (stop words removed) matrix shape: {tfidf_matrix_sw.shape}")

# Compare features in first document before and after stop word removal
tfidf_features_sw = tfidf_matrix_sw[0].nonzero()[1]
print(f"Number of non-zero features in first doc (TF-IDF, stop words removed): {len(tfidf_features_sw)}")
print(f"Vocabulary size change: {len(tfidf_vectorizer.vocabulary_)} -> {len(tfidf_vectorizer_sw.vocabulary_)}")

# Task 5: New Document Processing
def process_new_document(document, vectorizer):
    """
    Process a new document using trained vectorizer
    
    Args:
        document (str): Text to process
        vectorizer: Trained CountVectorizer or TfidfVectorizer
    
    Returns:
        sparse matrix: Document vector
    """
    return vectorizer.transform([document])

# Test with a sample news article
sample_news = "The stock market rallied as investors reacted to positive economic data and central bank announcements."
bow_sample = process_new_document(sample_news, count_vectorizer)
tfidf_sample = process_new_document(sample_news, tfidf_vectorizer)
tfidf_sample_sw = process_new_document(sample_news, tfidf_vectorizer_sw)

print("\nTest document analysis:")
print(f"Non-zero features (BoW): {bow_sample.nnz}")
print(f"Non-zero features (TF-IDF): {tfidf_sample.nnz}")
print(f"Non-zero features (TF-IDF, stop words removed): {tfidf_sample_sw.nnz}")

# Show terms found in test document (stop words removed)
sample_indices = tfidf_sample_sw.nonzero()[1]
sample_terms = [tfidf_vectorizer_sw.get_feature_names_out()[i] for i in sample_indices]
print("Terms found in test document (stop words removed):", sample_terms)

# Analysis Questions
print("\nAnalysis:")
print("1. Vocabulary sizes and matrix shapes indicate Reuters corpus is large and diverse.")
print("2. Non-zero features per document decrease after stop word removal, improving focus on meaningful terms.")
print("3. Terms not in training vocabulary are ignored in new document processing.")
print("4. Most common terms in test document are finance/economics related, reflecting domain relevance.")


TF-IDF vocabulary size: 30916
TF-IDF matrix shape: (10788, 30916)
Number of non-zero features in first doc (BoW): 351
Number of non-zero features in first doc (TF-IDF): 351

TF-IDF (stop words removed) vocabulary size: 30627
TF-IDF (stop words removed) matrix shape: (10788, 30627)
Number of non-zero features in first doc (TF-IDF, stop words removed): 269
Vocabulary size change: 30916 -> 30627

Test document analysis:
Non-zero features (BoW): 15
Non-zero features (TF-IDF): 15
Non-zero features (TF-IDF, stop words removed): 11
Terms found in test document (stop words removed): ['announcements', 'bank', 'central', 'data', 'economic', 'investors', 'market', 'positive', 'rallied', 'reacted', 'stock']

Analysis:
1. Vocabulary sizes and matrix shapes indicate Reuters corpus is large and diverse.
2. Non-zero features per document decrease after stop word removal, improving focus on meaningful terms.
3. Terms not in training vocabulary are ignored in new document processing.
4. Most common ter